In [ ]:
import os; os.makedirs('/tmp/wk', exist_ok=True)
open('/tmp/wk/finetune_cookbook.sh', 'w').write('#!/bin/bash\n# Vintern-1B-v3_5 2nd-stage fine-tune on AutoViVQA (grouped leak-free split).\n#\n# VERBATIM from the official Vintern cookbook recipe\n#   (reference/repo_default_finetune_lora.sh + colab cells 35-38):\n#   freeze backbone + MLP + LLM ; LoRA rank 16 on the LLM only ;\n#   max_dynamic_patch 6 ; force_image_size 448 ; down_sample_ratio 0.5 ;\n#   lr 4e-5 ; cosine ; warmup 0.03 ; wd 0.01 ; 1 epoch ; conv_style Hermes-2.\n#\n# Only deviations: paths, GPU count, and total-batch plumbing for a single\n# 16 GB card. Recipe hyper-params are UNCHANGED.\nset -x\n\nGPUS=${GPUS:-1}\nBATCH_SIZE=${BATCH_SIZE:-16}                     # cookbook total batch = 16\nPER_DEVICE_BATCH_SIZE=${PER_DEVICE_BATCH_SIZE:-4}\nGRADIENT_ACC=$((BATCH_SIZE / PER_DEVICE_BATCH_SIZE / GPUS))\n\nMODEL_PATH=${MODEL_PATH:-"./pretrained/Vintern-1B-v3_5"}\nMETA_PATH=${META_PATH:-"./shell/data/meta_autovivqa.json"}\nOUTPUT_DIR=${OUTPUT_DIR:-"work_dirs/vintern_1b_v3_5_autovivqa_lora"}\nEPOCHS=${EPOCHS:-1}\nSEED=${SEED:-42}\nRESUME_ARG=${RESUME_ARG:-}                       # e.g. "--resume_from_checkpoint <dir>/checkpoint-1000"\n\n# --overwrite_output_dir would wipe a resume checkpoint we just copied in.\nOVERWRITE=True\n[ -n "$RESUME_ARG" ] && OVERWRITE=False\n\nexport PYTHONPATH="${PYTHONPATH}:$(pwd)"\nexport MASTER_PORT=34229\nexport TF_CPP_MIN_LOG_LEVEL=3\nexport LAUNCHER=pytorch\n\nmkdir -p "$OUTPUT_DIR"\n\n# deepspeed is optional for a single-GPU LoRA run; drop it if unavailable\nDS_ARG="--deepspeed zero_stage1_config.json"\nif [ "${SKIP_DEEPSPEED:-0}" = "1" ] || [ ! -f "zero_stage1_config.json" ]; then\n  DS_ARG=""\n  echo "[finetune] deepspeed disabled (SKIP_DEEPSPEED=${SKIP_DEEPSPEED:-0}, config present=$([ -f zero_stage1_config.json ] && echo yes || echo no))"\nfi\n\ntorchrun \\\n  --nnodes=1 --node_rank=0 --master_addr=127.0.0.1 \\\n  --nproc_per_node=${GPUS} --master_port=${MASTER_PORT} \\\n  internvl/train/internvl_chat_finetune.py \\\n  --model_name_or_path "${MODEL_PATH}" \\\n  --conv_style "Hermes-2" \\\n  --output_dir ${OUTPUT_DIR} \\\n  --meta_path "${META_PATH}" \\\n  --overwrite_output_dir ${OVERWRITE} \\\n  ${RESUME_ARG} \\\n  --force_image_size 448 \\\n  --max_dynamic_patch 6 \\\n  --down_sample_ratio 0.5 \\\n  --drop_path_rate 0.0 \\\n  --freeze_llm True \\\n  --freeze_mlp True \\\n  --freeze_backbone True \\\n  --use_llm_lora 16 \\\n  --vision_select_layer -1 \\\n  --dataloader_num_workers 4 \\\n  --bf16 True \\\n  --seed ${SEED} \\\n  --num_train_epochs ${EPOCHS} \\\n  --per_device_train_batch_size ${PER_DEVICE_BATCH_SIZE} \\\n  --gradient_accumulation_steps ${GRADIENT_ACC} \\\n  --evaluation_strategy "no" \\\n  --save_strategy "steps" \\\n  --save_steps 500 \\\n  --save_total_limit 2 \\\n  --learning_rate 4e-5 \\\n  --weight_decay 0.01 \\\n  --warmup_ratio 0.03 \\\n  --lr_scheduler_type "cosine" \\\n  --logging_steps 10 \\\n  --max_seq_length 700 \\\n  --do_train True \\\n  --grad_checkpoint True \\\n  --group_by_length True \\\n  --dynamic_image_size True \\\n  --use_thumbnail True \\\n  --ps_version \'v2\' \\\n  ${DS_ARG} \\\n  --report_to "tensorboard" \\\n  2>&1 | tee -a "${OUTPUT_DIR}/training_log.txt"\n')
print('wrote /tmp/wk/finetune_cookbook.sh', os.path.getsize('/tmp/wk/finetune_cookbook.sh'), 'bytes')

In [ ]:
import os; os.makedirs('/tmp/wk', exist_ok=True)
open('/tmp/wk/merge_lora.py', 'w').write('"""Merge LoRA adapters into the base weights — verbatim from the Vintern\nfine-tune cookbook (colab cell 41 == Vintern repo tools/merge_lora.py).\n\n    python merge_lora.py <lora_output_dir> <merged_output_dir>\n\nRun from inside the Vintern repo\'s `internvl_chat/` directory (it imports\n`internvl.model.internvl_chat`).\n"""\nimport argparse\nimport sys\n\nimport torch\n\nsys.path.append("/tmp/wk/Vintern/internvl_chat")\nsys.path.append(".")\n\nfrom internvl.model.internvl_chat import InternVLChatModel  # noqa: E402\nfrom transformers import AutoTokenizer  # noqa: E402\n\nap = argparse.ArgumentParser()\nap.add_argument("input_path", type=str)\nap.add_argument("output_path", type=str)\nargs = ap.parse_args()\n\nprint("Loading model...")\nmodel = InternVLChatModel.from_pretrained(\n    args.input_path, low_cpu_mem_usage=True, torch_dtype=torch.bfloat16).eval()\nprint("Loading tokenizer...")\ntokenizer = AutoTokenizer.from_pretrained(args.input_path, trust_remote_code=True)\n\nif model.config.use_backbone_lora:\n    model.vision_model.merge_and_unload()\n    model.vision_model = model.vision_model.model\n    model.config.use_backbone_lora = 0\nif model.config.use_llm_lora:\n    model.language_model.merge_and_unload()\n    model.language_model = model.language_model.model\n    model.config.use_llm_lora = 0\n\nprint("Saving model...")\nmodel.save_pretrained(args.output_path)\nprint("Saving tokenizer...")\ntokenizer.save_pretrained(args.output_path)\nprint("Done!")\n')
print('wrote /tmp/wk/merge_lora.py', os.path.getsize('/tmp/wk/merge_lora.py'), 'bytes')

In [ ]:
import os; os.makedirs('/tmp/wk', exist_ok=True)
open('/tmp/wk/gen_vintern_standalone.py', 'w').write('"""Standalone generation for the Kaggle kernel (no repo clone, no metrics dep).\n\nGreedy `InternVLChatModel.chat` over a split, writes a bridge-pipeline-format\nprediction JSON. Scoring happens locally after fetch (score_local.py, which\nuses metrics.compute_score.compute_all_data -- the same function behind\nTable 1, so the number is apples-to-apples).\n\n    python gen_vintern_standalone.py --model-path <merged_dir> \\\n        --data <autovivqa_val.jsonl> --images-dir <dir> --out <out_dir> --max-num 6\n"""\nimport argparse\nimport json\nimport os\nimport sys\n\nimport torch\nimport torchvision.transforms as T\nfrom PIL import Image\nfrom torchvision.transforms.functional import InterpolationMode\n\nsys.path.append("/tmp/wk/Vintern/internvl_chat")\nfrom internvl.model.internvl_chat import InternVLChatModel  # noqa: E402\nfrom transformers import AutoTokenizer  # noqa: E402\n\nIMAGENET_MEAN = (0.485, 0.456, 0.406)\nIMAGENET_STD = (0.229, 0.224, 0.225)\n\n\ndef build_transform(sz):\n    return T.Compose([\n        T.Lambda(lambda im: im.convert("RGB") if im.mode != "RGB" else im),\n        T.Resize((sz, sz), interpolation=InterpolationMode.BICUBIC),\n        T.ToTensor(), T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)])\n\n\ndef closest_ratio(ar, ratios, w, h, sz):\n    best_d, best = float("inf"), (1, 1)\n    area = w * h\n    for r in ratios:\n        d = abs(ar - r[0] / r[1])\n        if d < best_d:\n            best_d, best = d, r\n        elif d == best_d and area > 0.5 * sz * sz * r[0] * r[1]:\n            best = r\n    return best\n\n\ndef dynamic_preprocess(image, min_num=1, max_num=6, image_size=448, use_thumbnail=True):\n    w, h = image.size\n    ar = w / h\n    ratios = sorted({(i, j) for n in range(min_num, max_num + 1) for i in range(1, n + 1)\n                     for j in range(1, n + 1) if min_num <= i * j <= max_num}, key=lambda x: x[0] * x[1])\n    tr = closest_ratio(ar, ratios, w, h, image_size)\n    tw, th = image_size * tr[0], image_size * tr[1]\n    blocks = tr[0] * tr[1]\n    resized = image.resize((tw, th))\n    tiles = [resized.crop((\n        (i % (tw // image_size)) * image_size, (i // (tw // image_size)) * image_size,\n        ((i % (tw // image_size)) + 1) * image_size, ((i // (tw // image_size)) + 1) * image_size))\n        for i in range(blocks)]\n    if use_thumbnail and len(tiles) != 1:\n        tiles.append(image.resize((image_size, image_size)))\n    return tiles\n\n\ndef load_image(path, sz=448, max_num=6):\n    im = Image.open(path).convert("RGB")\n    tf = build_transform(sz)\n    return torch.stack([tf(t) for t in dynamic_preprocess(im, image_size=sz, max_num=max_num)])\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--model-path", required=True)\n    ap.add_argument("--data", required=True)\n    ap.add_argument("--images-dir", required=True)\n    ap.add_argument("--out", required=True)\n    ap.add_argument("--max-num", type=int, default=6)\n    ap.add_argument("--max-new-tokens", type=int, default=64)\n    ap.add_argument("--limit", type=int, default=0)\n    a = ap.parse_args()\n\n    outp = os.path.join(a.out, "results")\n    os.makedirs(outp, exist_ok=True)\n\n    tok = AutoTokenizer.from_pretrained(a.model_path, trust_remote_code=True, use_fast=False)\n    model = InternVLChatModel.from_pretrained(\n        a.model_path, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True).eval().cuda()\n    model.img_context_token_id = tok.convert_tokens_to_ids("<IMG_CONTEXT>")\n\n    gcfg = dict(max_new_tokens=a.max_new_tokens, do_sample=False, num_beams=1)\n    rows = [json.loads(l) for l in open(a.data, encoding="utf-8")]\n    if a.limit:\n        rows = rows[: a.limit]\n\n    samples = []\n    for i, r in enumerate(rows):\n        pv = load_image(os.path.join(a.images_dir, r["image"]), max_num=a.max_num).to(torch.bfloat16).cuda()\n        q = r["conversations"][0]["value"]\n        try:\n            pred = model.chat(tok, pv, q, gcfg)\n        except Exception as e:  # noqa\n            pred = f"[gen-error: {str(e)[:100]}]"\n        samples.append({"index": i, "question": q.replace("<image>\\n", ""),\n                        "prediction": pred.strip() if isinstance(pred, str) else str(pred),\n                        "ground_truths": r["all_answers"]})\n        if (i + 1) % 200 == 0:\n            print(f"  {i + 1}/{len(rows)}", flush=True)\n\n    pf = os.path.join(outp, "text_predictions_epoch_1.json")\n    json.dump({"epoch": 1, "samples": samples}, open(pf, "w"), ensure_ascii=False, indent=1)\n    print(f"n={len(samples)} -> {pf}")\n\n\nif __name__ == "__main__":\n    main()\n')
print('wrote /tmp/wk/gen_vintern_standalone.py', os.path.getsize('/tmp/wk/gen_vintern_standalone.py'), 'bytes')

In [ ]:
import subprocess
subprocess.call('git clone -q --depth 1 https://github.com/5CD-AI/Vintern.git /tmp/wk/Vintern || (mkdir -p /tmp/wk && cd /tmp/wk && git clone -q --depth 1 https://github.com/5CD-AI/Vintern.git Vintern)', shell=True)
# InternVL patch/__init__ hard-imports flash_attn monkey-patches we don't need
# (Kaggle GPUs are pre-Ampere -> can't run flash-attn v2 anyway); strip the 2 lines.
import os
os.system("sed -i '/flash_attn_monkey_patch import/d' /tmp/wk/Vintern/internvl_chat/internvl/patch/__init__.py")
os.system('ls /tmp/wk/Vintern/internvl_chat && head -4 /tmp/wk/Vintern/internvl_chat/internvl/patch/__init__.py')

In [ ]:
# cookbook env, in full, with the two Kaggle-2026-compat pins we proved necessary:
#  - torch 2.5.1 (Kaggle's default torch is too new for transformers 4.47 / this 2024 trainer)
#  - deepspeed==0.15.4, peft==0.14.0 (latest deepspeed/peft break on torch 2.5's schema infer
#    and Kaggle's old torchao respectively) -- installed IMPORT-ONLY (DS_BUILD_OPS=0), the
#    trainer runs plain (no --deepspeed arg, single GPU LoRA doesn't need ZeRO)
#  - NO flash_attn install (P100/T4 can't run it; every model file gates it behind try/except)
!pip -q install torch==2.5.1 torchvision==0.20.1 --index-url https://download.pytorch.org/whl/cu121
!DS_BUILD_OPS=0 pip -q install timm einops 'peft==0.14.0' wandb deepspeed==0.15.4 bitsandbytes decord tensorboardX gdown imageio opencv-python-headless
!pip -q install -U datasets
!pip -q install transformers==4.47.0 'accelerate>=1.1,<1.3' 'numpy<2.1'
!pip -q uninstall -y torchao 2>/dev/null; echo done

In [ ]:
import subprocess, sys, os
os.chdir('/tmp/wk/Vintern/internvl_chat')
r = subprocess.run([sys.executable, '-c',
  'import torch,transformers,deepspeed,decord,timm,peft,cv2,imageio;'
  'import internvl.patch, internvl.train.dataset, internvl.model.internvl_chat;'
  'from internvl.train.trainer_monkey_patch import replace_create_optimizer;'
  'print("OK", torch.__version__, transformers.__version__)'],
  capture_output=True, text=True)
print(r.stdout); print(r.stderr)
assert 'OK' in r.stdout, 'import chain broken'

In [ ]:
import os
os.chdir('/tmp/wk/Vintern')
!mkdir -p pretrained && huggingface-cli download --resume-download --local-dir-use-symlinks False 5CD-AI/Vintern-1B-v3_5 --local-dir pretrained/Vintern-1B-v3_5 2>&1 | tail -3

In [ ]:
# our pre-built splits (already committed as a tiny Kaggle dataset -- no repo clone needed)
import json, os
dst = '/tmp/wk/Vintern/internvl_chat/shell/data'; os.makedirs(dst, exist_ok=True)
meta = {'autovivqa-train': {'root': '/kaggle/input/auto-vqabest/preprocessed_images',
  'annotation': '/kaggle/input/autovivqa-internvl-sft/autovivqa_train.jsonl', 'data_augment': False, 'repeat_time': 1,
  'length': sum(1 for _ in open('/kaggle/input/autovivqa-internvl-sft/autovivqa_train.jsonl'))}}
json.dump(meta, open(f'{dst}/meta_autovivqa.json', 'w'), ensure_ascii=False, indent=2)
print(meta)

In [ ]:
%cd /tmp/wk/Vintern/internvl_chat
import os, torch
ng = torch.cuda.device_count()
print('GPUs:', ng, [torch.cuda.get_device_name(i) for i in range(ng)])
os.environ['GPUS'] = str(ng)
os.environ['BATCH_SIZE'] = '16'
os.environ['PER_DEVICE_BATCH_SIZE'] = '4' if ng >= 2 else '2'
os.environ['PYTHONPATH'] = os.getcwd()
os.environ['MODEL_PATH'] = '/tmp/wk/Vintern/pretrained/Vintern-1B-v3_5'
os.environ['META_PATH'] = 'shell/data/meta_autovivqa.json'
os.environ['OUTPUT_DIR'] = '/kaggle/working/work_dirs/vintern_lora'
os.environ['SEED'] = '42'
os.environ['EPOCHS'] = '1'
os.environ['SKIP_DEEPSPEED'] = '1'
os.environ['RESUME_ARG'] = ''
print('RESUME_ARG=', os.environ['RESUME_ARG'])
!bash /tmp/wk/finetune_cookbook.sh 2>&1 | tail -80
print('checkpoints now:', sorted(os.listdir('/kaggle/working/work_dirs/vintern_lora')) if os.path.isdir('/kaggle/working/work_dirs/vintern_lora') else 'MISSING')

In [ ]:
import os, glob
done = os.path.exists('/kaggle/working/work_dirs/vintern_lora/adapter_model.safetensors') or os.path.exists('/kaggle/working/work_dirs/vintern_lora/pytorch_model.bin')
print('epoch finished (final adapter present)?', done)
os.environ['EPOCH_DONE'] = '1' if done else '0'

In [ ]:
if os.environ.get('EPOCH_DONE') == '1':
    !python /tmp/wk/merge_lora.py /kaggle/working/work_dirs/vintern_lora /kaggle/working/work_dirs/vintern_lora_merge
    !cp /tmp/wk/Vintern/pretrained/Vintern-1B-v3_5/*.py /kaggle/working/work_dirs/vintern_lora_merge/
    !cp /tmp/wk/Vintern/pretrained/Vintern-1B-v3_5/config.json /kaggle/working/work_dirs/vintern_lora_merge/
    !ls /kaggle/working/work_dirs/vintern_lora_merge
else:
    print('skip merge/generate -- epoch not finished yet, checkpoint will be promoted for resume')

In [ ]:
if os.environ.get('EPOCH_DONE') == '1':
    import sys; sys.path.append('/tmp/wk')
    import subprocess
    subprocess.run(['python', '/tmp/wk/gen_vintern_standalone.py', '--model-path', '/kaggle/working/work_dirs/vintern_lora_merge',
      '--data', '/kaggle/input/autovivqa-internvl-sft/autovivqa_val.jsonl', '--images-dir', '/kaggle/input/auto-vqabest/preprocessed_images',
      '--out', '/kaggle/working/out/val', '--max-num', '6'])

In [ ]:
if os.environ.get('EPOCH_DONE') == '1':
    import subprocess
    subprocess.run(['python', '/tmp/wk/gen_vintern_standalone.py', '--model-path', '/kaggle/working/work_dirs/vintern_lora_merge',
      '--data', '/kaggle/input/autovivqa-internvl-sft/autovivqa_test.jsonl', '--images-dir', '/kaggle/input/auto-vqabest/preprocessed_images',
      '--out', '/kaggle/working/out/test', '--max-num', '6'])